In [13]:
from nuevo_analysis import *
load history.pkl

datos = extraer_activaciones()                  # forward pass + guarda .pkl
analisis_desempeno(datos, history=history)      # curvas + matriz confusión
visualizar_arcos_emocionales(datos)             # arco real vs predicho
analizar_dependencias_tropo(datos)              # correlaciones temporales
analizar_atencion(datos)                        # en qué se fija la red
pca_hidden_states(datos)                        # PCA + t-SNE
perfil_emocional_por_tropo(datos)              # firma emocional por tropo
similitud_arco_respuesta(datos)                # congruencia arco-lector
analisis_errores(datos)                         # qué confunde el modelo

SyntaxError: invalid syntax (2519765168.py, line 2)

In [12]:
import os
import pickle
import torch
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from sklearn.decomposition import PCA
from torch.utils.data import DataLoader
from tqdm import tqdm

import configl
from model import JointModel
from train import FanficDataset, create_dataloaders

def generar_datos_analisis(pkl_path="data/features/emotional_features.pkl"):
    # 1. Cargar datos y modelo
    with open(pkl_path, "rb") as f:
        features = pickle.load(f)
    
    ds = FanficDataset(features)
    _, _, test_dl = create_dataloaders(ds, batch_size=1)
    
    device = "cuda" if torch.cuda.is_available() else "cpu"
    model = JointModel()
    model.load_state_dict(torch.load("models/best.pt", map_location=device))
    model.to(device)
    model.eval()
    
    resultados_analisis = []
    
    print("🔮 Extrayendo activaciones e intensidades de la red para el set de Test...")
    with torch.no_grad():
        # El bucle va aquí, DENTRO de la función y del with torch.no_grad()
        for batch in tqdm(test_dl, desc="Extrayendo activaciones"):
            win_pad = batch[0].to(device)
            win_counts = batch[1].to(device)
            com_pad = batch[2].to(device)
            lengths = batch[6].item()
            
            outputs = model(
                win_pad, win_counts, com_pad, 
                return_attention=True, 
                return_hidden=True
            )
            
            hidden_states = outputs["hidden_states"][0, :lengths, :].cpu().numpy()
            attn_weights = outputs["attention_weights"][0, :lengths].cpu().numpy()
            reader_pred = outputs["reader_pred"][0, :lengths, :].cpu().numpy()
            reader_ground_truth = batch[4][0, :lengths, :].cpu().numpy()
            trope_real = batch[3].item()
            trope_pred = torch.argmax(outputs["trope_logits"], dim=1).item()
            
            resultados_analisis.append({
                "num_chapters": lengths,
                "trope_real": configl.IDX_TO_TROPE[trope_real],
                "trope_pred": configl.IDX_TO_TROPE[trope_pred],
                "hidden_states": hidden_states,
                "attention_weights": attn_weights,
                "reader_emotions_pred": reader_pred,
                "reader_emotions_real": reader_ground_truth
            })
            
    # Guardar resultados
    os.makedirs(configl.RESULTS_DIR, exist_ok=True)
    out_path = os.path.join(configl.RESULTS_DIR, "datos_visualizacion.pkl")
    with open(out_path, "wb") as f:
        pickle.dump(resultados_analisis, f)
    print(f"✓ Datos exportados exitosamente a {out_path}")

if __name__ == "__main__":
    generar_datos_analisis()

import pickle
import matplotlib.pyplot as plt
import seaborn as sns
import configl

with open("results/datos_visualizacion.pkl", "rb") as f:
    datos = pickle.load(f)

# Elegimos un fanfic de ejemplo (el primero del archivo)
fanfic = datos[0]
caps = range(1, fanfic["num_chapters"] + 1)

# Elegir qué emociones de GoEmotions de la lista de 28 quieres graficar
# Índices de ejemplo basados en configl.EMOTION_LABELS (asumiendo tristeza y alegría)
idx_sadness = configl.EMOTION_LABELS.index("sadness")
idx_joy = configl.EMOTION_LABELS.index("joy")

plt.figure(figsize=(12, 5))

# Graficar la Realidad de los Comentarios (Ground Truth) vs la Predicción del modelo
plt.plot(caps, fanfic["reader_emotions_real"][:, idx_sadness], 'g-', label="Tristeza Real (Comentarios)", marker='o')
plt.plot(caps, fanfic["reader_emotions_pred"][:, idx_sadness], 'g--', label="Tristeza Predicha (Modelo)", marker='x')

plt.plot(caps, fanfic["reader_emotions_real"][:, idx_joy], 'b-', label="Alegría Real (Comentarios)", marker='o')
plt.plot(caps, fanfic["reader_emotions_pred"][:, idx_joy], 'b--', label="Alegría Predicha (Modelo)", marker='x')

plt.title(f"Arco Emocional por Capítulo — Trope Real: {fanfic['trope_real']} (Predicción: {fanfic['trope_pred']})")
plt.xlabel("Número de Capítulo")
plt.ylabel("Intensidad Emocional")
plt.xticks(caps)
plt.legend()
plt.grid(True, linestyle=":")
plt.show()



import numpy as np

# Vamos a acumular las atenciones de varias obras alineándolas por porcentaje de avance 
# (ya que los fanfics tienen longitudes variables de 1 a 15 capítulos)
atenciones_por_trope = {trope: [] for trope in configl.TROPES}

for f in datos:
    # Para promediar limpiamente obras cortas y largas, interpolamos a un vector fijo de 10 puntos (deciles de la obra)
    pesos = f["attention_weights"]
    xp = np.linspace(0, 1, len(pesos))
    x_nuevo = np.linspace(0, 1, 10)
    pesos_normalizados = np.interp(x_nuevo, xp, pesos)
    # Asegurar que sigan sumando 1
    pesos_normalizados /= pesos_normalizados.sum()
    
    atenciones_por_trope[f["trope_real"]].append(pesos_normalizados)

# Crear la matriz promedio para el mapa de calor
matriz_calor = []
for trope in configl.TROPES:
    matriz_calor.append(np.mean(atenciones_por_trope[trope], axis=0))

plt.figure(figsize=(10, 4))
sns.heatmap(matriz_calor, annot=True, cmap="YlOrRd", yticklabels=configl.TROPES,
            xticklabels=[f"{i*10}%" for i in range(1, 11)])
plt.title("¿En qué parte de la obra se fija el Modelo? (Distribución de la Atención)")
plt.xlabel("Progreso a lo largo del Fanfic (Capítulos Normalizados)")
plt.ylabel("Trope Literario")
plt.show()


from sklearn.decomposition import PCA
import pandas as pd

X_hidden = []
y_tropes = []

# Extraemos cada capítulo como un punto individual en nuestro espacio latente
for f in datos:
    for cap_hidden in f["hidden_states"]:
        X_hidden.append(cap_hidden)
        y_tropes.append(f["trope_real"])

X_hidden = np.array(X_hidden)

# Ajustar PCA a 2 dimensiones
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_hidden)

# Crear Dataframe para graficar con Seaborn de forma elegante
df_pca = pd.DataFrame({
    "Componente Principal 1": X_pca[:, 0],
    "Componente Principal 2": X_pca[:, 1],
    "Trope": y_tropes
})

plt.figure(figsize=(9, 7))
sns.scatterplot(
    data=df_pca, 
    x="Componente Principal 1", 
    y="Componente Principal 2", 
    hue="Trope", 
    palette="Set2",
    alpha=0.7,
    edgecolor='none'
)
plt.title("Proyección PCA del Estado Oculto de la LSTM (Capítulos en el Espacio Latente)")
plt.grid(True, alpha=0.3)
plt.show()

print(f"Varianza explicada por los 2 primeros componentes: {sum(pca.explained_variance_ratio_)*100:.2f}%")

🔮 Extrayendo activaciones e intensidades de la red para el set de Test...


Extrayendo activaciones:   0%|                                                                              | 0/41 [00:00<?, ?it/s]


KeyError: 0